<a href="https://colab.research.google.com/github/asiahcrutchfield/ground-level-languages-ai/blob/main/gll_intro_to_ai.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ==============================================================================
# Ground Level Languages — Update #1 Acoustic Pipeline (Native Oral Posture)
# Models: Allosaurus (Phone Detection) + Edge Neural TTS (Native IPA) + TorchCREPE
# ==============================================================================

!pip install -q allosaurus torchcrepe edge-tts nest_asyncio librosa soundfile scipy numpy

import os
import json
import zipfile
import asyncio
import nest_asyncio
import edge_tts
import numpy as np
import torch
import torchcrepe
import librosa
import soundfile as sf
from scipy.signal import find_peaks
from allosaurus.app import read_recognizer

nest_asyncio.apply()

# 1. Create output directories
os.makedirs("audio/slices", exist_ok=True)
os.makedirs("audio/phonemes", exist_ok=True)

def apply_fade(clip, sr=16000, fade_ms=12):
    """Smooth fade-in/fade-out so isolated sounds never click or clip."""
    fade_len = min(int(sr * (fade_ms / 1000.0)), len(clip) // 2)
    if fade_len <= 1:
        return clip
    win = np.ones(len(clip), dtype=np.float32)
    win[:fade_len] = np.linspace(0.0, 1.0, fade_len)
    win[-fade_len:] = np.linspace(1.0, 0.0, fade_len)
    return clip * win

# 2. Native en-US Neural Phoneme Bank (authentic English oral posture, zero "ga-a-ga")
NATIVE_VOICE = "en-US-AvaNeural"

# Maps Allosaurus IPA detections -> (output_filename, IPA_for_SSML, max_duration_sec)
ENG_NATIVE_PHONEMES = {
    "k":  ("k.wav",  "k",  0.14),
    "kʰ": ("k.wav",  "k",  0.14),
    "t":  ("t.wav",  "t",  0.14),
    "tʰ": ("t.wav",  "t",  0.14),
    "p":  ("p.wav",  "p",  0.14),
    "d":  ("d.wav",  "d",  0.12),
    "ɡ":  ("g.wav",  "ɡ",  0.12),
    "g":  ("g.wav",  "ɡ",  0.12),
    "dʒ": ("dz.wav", "dʒ", 0.15),
    "w":  ("w.wav",  "w",  0.16),
    "l":  ("l.wav",  "l",  0.22),
    "r":  ("r.wav",  "ɹ",  0.22),
    "ɹ":  ("r.wav",  "ɹ",  0.22),
    "m":  ("m.wav",  "m",  0.22),
    "æ":  ("ae.wav", "æ",  0.25),
    "ɑ":  ("aa.wav", "ɑ",  0.25),
    "ɔ":  ("ao.wav", "ɔ",  0.25),
    "ʌ":  ("ah.wav", "ʌ",  0.22),
    "i":  ("iy.wav", "iː", 0.25),
    "iː": ("iy.wav", "iː", 0.25),
}

async def generate_native_phonemes():
    print(f"Synthesizing crisp native oral-posture phonemes using {NATIVE_VOICE}...")
    for ipa_key, (fname, ssml_ipa, max_dur) in ENG_NATIVE_PHONEMES.items():
        out_path = f"audio/phonemes/{fname}"
        if os.path.exists(out_path):
            continue
        temp_mp3 = f"/content/tmp_{fname}.mp3"
        ssml = (
            f'<speak version="1.0" xmlns="http://www.w3.org/2001/10/synthesis" xml:lang="en-US">'
            f'<voice name="{NATIVE_VOICE}"><phoneme alphabet="ipa" ph="{ssml_ipa}">a</phoneme></voice>'
            f'</speak>'
        )
        communicate = edge_tts.Communicate(ssml, NATIVE_VOICE)
        await communicate.save(temp_mp3)

        clip, _ = librosa.load(temp_mp3, sr=16000)
        clip, _ = librosa.effects.trim(clip, top_db=32)
        clip = apply_fade(clip[:int(max_dur * 16000)], sr=16000, fade_ms=12)
        sf.write(out_path, clip, 16000)

asyncio.run(generate_native_phonemes())
print("Native phoneme bank ready in audio/phonemes/!\n")

# 3. Load Allosaurus & TorchCREPE models
recognizer = read_recognizer()
device = "cuda" if torch.cuda.is_available() else "cpu"
IPA_VOWELS = set("aeiouyɨʉɯuɪʏʊeøɘɵɤoəɛœɜɞʌɔæɐaɶɑɒɚɝ")

def classify_phone(ipa_symbol):
    return "vowel" if any(ch in IPA_VOWELS for ch in ipa_symbol) else "consonant"

def synthesize_humming_track(y, f0, periodicity, sr=16000, hop_length=160):
    """Generates a pure vocal humming track gated by speech energy + CREPE F0."""
    total_samples = len(y)
    freq_interp = np.interp(np.arange(total_samples), np.linspace(0, total_samples, len(f0)), f0)
    conf_interp = np.interp(np.arange(total_samples), np.linspace(0, total_samples, len(periodicity)), periodicity)

    rms = librosa.feature.rms(y=y, frame_length=512, hop_length=hop_length)[0]
    rms_interp = np.interp(np.arange(total_samples), np.linspace(0, total_samples, len(rms)), rms)
    rms_norm = rms_interp / (np.max(rms_interp) + 1e-6)

    voiced_mask = (conf_interp > 0.15) & (rms_norm > 0.12)
    amp = np.where(voiced_mask, 0.35 * rms_norm, 0.0)

    phase = (2.0 * np.pi * np.cumsum(freq_interp)) / sr
    return (amp * (0.8 * np.sin(phase) + 0.2 * np.sin(2.0 * phase))).astype(np.float32)

def process_word(word_id, mp3_path, visual_path):
    y_raw, sr = librosa.load(mp3_path, sr=16000)
    y, _ = librosa.effects.trim(y_raw, top_db=28)
    y = np.pad(y, (int(0.04 * sr), int(0.04 * sr)), mode="constant")

    temp_wav = f"/content/{word_id}_16k.wav"
    sf.write(temp_wav, y, sr)

    # STEP 1: Allosaurus phone detection -> Native oral-posture phoneme bank
    raw_phones = recognizer.recognize(temp_wav, lang_id='eng', timestamp=True, topk=1)
    detected_list = [line.split()[2] for line in raw_phones.strip().split("\n") if line.strip()]
    print(f"[{word_id}] Allosaurus detected sequence: {detected_list}")

    phonemes_out = []
    for idx, ipa in enumerate(detected_list):
        if ipa in ENG_NATIVE_PHONEMES:
            audio_url = f"audio/phonemes/{ENG_NATIVE_PHONEMES[ipa][0]}"
        else:
            print(f"  -> Notice: '{ipa}' not in ENG_NATIVE_PHONEMES, falling back to full syllable.")
            audio_url = f"audio/slices/{word_id}_s1.wav"

        phonemes_out.append({
            "id": f"{word_id}_p{idx+1}",
            "kind": classify_phone(ipa),  # "consonant" or "vowel" (zero IPA text sent to UI)
            "audio_url": audio_url
        })

    # STEP 2: Syllable duration segmentation
    rms = librosa.feature.rms(y=y, frame_length=512, hop_length=160)[0]
    peaks, _ = find_peaks(rms, distance=12, prominence=np.max(rms) * 0.2)
    if len(peaks) == 0:
        peaks = [len(rms) // 2]

    boundaries = [0] + [int((peaks[i] + peaks[i+1]) / 2) for i in range(len(peaks) - 1)] + [len(rms)]
    syllables_out = []
    for idx in range(len(boundaries) - 1):
        s_s = (boundaries[idx] * 160) / sr
        e_s = (boundaries[idx+1] * 160) / sr
        dur = max(0.08, e_s - s_s)
        clip_path = f"audio/slices/{word_id}_s{idx+1}.wav"
        sf.write(clip_path, apply_fade(y[int(s_s * sr):int(e_s * sr)], sr=sr), sr)
        syllables_out.append({
            "id": f"{word_id}_s{idx+1}",
            "width_px": int(min(180, max(65, dur * 240))),
            "audio_url": clip_path
        })

    # STEP 3: TorchCREPE F0 Pitch Extraction & Stripped Humming Synthesis
    pitch, periodicity = torchcrepe.predict(
        torch.tensor(y).unsqueeze(0), sr, hop_length=160, fmin=65.0, fmax=500.0,
        model="full", decoder=torchcrepe.decode.viterbi, return_periodicity=True, device=device
    )
    frequency = pitch.squeeze(0).cpu().numpy()
    conf = periodicity.squeeze(0).cpu().numpy()

    hum_out = f"audio/slices/{word_id}_hum.wav"
    sf.write(hum_out, synthesize_humming_track(y, frequency, conf, sr=sr), sr)

    tones_out = []
    for idx in range(len(syllables_out)):
        s_f = int((idx / len(syllables_out)) * len(frequency))
        e_f = int(((idx + 1) / len(syllables_out)) * len(frequency))
        slope = float(frequency[e_f-1] - frequency[s_f]) if e_f - s_f > 1 else 0.0
        style = "solid" if slope > 12 else ("ring" if slope < -12 else "dot")
        tones_out.append({"id": f"{word_id}_t{idx+1}", "style": style, "audio_url": hum_out})

    return {
        "id": word_id,
        "visual_url": visual_path,
        "full_audio_url": f"audio/{word_id}.mp3",
        "humming_audio_url": hum_out,
        "phonemes": phonemes_out,
        "syllables": syllables_out,
        "tones": tones_out
    }

# 4. Run on your 5 Level 0 words
words = [
    ("cat", "/content/cat.mp3", "images/visuals/cat.jpg"),
    ("dog", "/content/dog.mp3", "images/visuals/dog.jpg"),
    ("walk", "/content/walk.mp3", "images/visuals/walk.gif"),
    ("jump", "/content/jump.mp3", "images/visuals/jump.gif"),
    ("tree", "/content/tree.mp3", "images/visuals/tree.jpg")
]

dataset = {"items": [process_word(w, p, v) for w, p, v in words if os.path.exists(p)]}

with open("lesson_data.json", "w") as f:
    json.dump(dataset, f, indent=2)

# 5. Zip lesson_data.json + audio/slices/ + audio/phonemes/
with zipfile.ZipFile("gll_update1_assets.zip", "w") as zf:
    zf.write("lesson_data.json")
    for folder in ["audio/slices", "audio/phonemes"]:
        for root, _, files in os.walk(folder):
            for file in files:
                zf.write(os.path.join(root, file))

print("\nDone! Download gll_update1_assets.zip and overwrite lesson_data.json and audio/ in your project.")

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Synthesizing crisp native oral-posture phonemes using en-US-AvaNeural...
Native phoneme bank ready in audio/phonemes/!

downloading model  latest
from:  https://github.com/xinjli/allosaurus/releases/download/v1.0/latest.tar.gz
to:    /usr/local/lib/python3.13/dist-packages/allosaurus/pretrained
please wait...

Done! Download gll_update1_assets.zip and overwrite lesson_data.json and audio/ in your project.
